##Model Training##
This notebook trains a teacher model, distills it into a student model, and exports the final model artifacts.  
The output includes the trained **.h5 model**, **.tflite model** for profiling.

##Set Project Path##
Add the directory containing `custom_model.py` to your system path so the notebook can import your model specifications and training scripts.

In [0]:
import sys

sys.path.append("<PATH_TO_DIRECTORY_CONTAINING_custom_model.py>")

##Install Required Dependencies##

In [0]:
pip install silabs-mltk audiomentations noisereduce pyloudnorm librosa tensorflow scikit-learn tensorflow-model-optimization

##Restart Python Interpreter##
Run this command to refresh the environment and ensure all newly installed libraries are correctly recognized by the notebook.

In [0]:
dbutils.library.restartPython()

##Train Teacher Model##

Trains the teacher model by **loading a base model if it exists**, otherwise **starting from scratch**. The `mltk_core` pipeline automatically resumes from an existing `custom_model_teacher.h5` or initializes a new model when none is found.

In [0]:
#Train Teacher Model
import os
os.environ["TRAIN_TEACHER"] = "1"  

import mltk.core as mltk_core
from custom_model import my_model, prepare_teacher_or_student_model

prepare_teacher_or_student_model(train_teacher=True)

base_model_path = "<PATH_TO_WORKSPACE_FILES>/models/custom_model_teacher.h5"
if os.path.exists(base_model_path):
    print(f"Resuming training from: {base_model_path}")
    results = mltk_core.train_model(my_model, clean=False, weights=base_model_path)
else:
    print("Starting training from scratch...")
    results = mltk_core.train_model(my_model, clean=True)

print(results)

## Train Student Model

Trains the student model by **loading a previous student checkpoint if available**, otherwise **starting from scratch**, and uses the fine‑tuned teacher model for knowledge distillation.

In [0]:
#Train Student Model
import os
os.environ["TRAIN_TEACHER"] = "0"  

import mltk.core as mltk_core
from custom_model import my_model, prepare_teacher_or_student_model

prepare_teacher_or_student_model(train_teacher=False)

base_model_path = "<PATH_TO_WORKSPACE_FILES>/models/custom_model.h5"
if os.path.exists(base_model_path):
    print(f"Resuming training from: {base_model_path}")
    results = mltk_core.train_model(my_model, clean=False)
else:
    print("Starting training from scratch...")
    results = mltk_core.train_model(my_model, clean=True)

print(results)

##Locate and Copy Model Artifacts##
This step searches for trained model artifacts inside driver nodes, identifies the correct `custom_model` folder, and copies the generated `.h5` and `.tflite` files into your Workspace for easy access.

In [0]:
import glob, os, pprint, shutil

# 1) Find all potential driver homes
driver_homes = [d.rstrip("/") for d in glob.glob("/home/spark-*/")]
if not driver_homes:
    raise RuntimeError("No /home/spark-*/ found on this cluster")

# 2) For each driver, look for a non-empty custom_model folder
candidates = []
for dh in driver_homes:
    cm_dir = f"{dh}/.mltk/models/custom_model"
    if os.path.isdir(cm_dir):
        try:
            files = os.listdir(cm_dir)
        except Exception:
            files = []
        candidates.append((cm_dir, files))

print("MLTK candidate dirs:")
pprint.pp(candidates)

# 3) Choose the one that actually has artifacts (h5 or tflite)
src_dir = None
for cm_dir, files in candidates:
    if any(f.endswith((".h5", ".tflite")) for f in files):
        src_dir = cm_dir
        break

if not src_dir:
    raise RuntimeError("No custom_model artifacts found. Re-run STUDENT training, then run this cell again.")

# 4) Copy selected artifacts to Workspace Files
#Create a folder in workspace files to store the models Eg: models

dst_dir = "<PATH_TO_WORKSPACE_FILES>/models>"
os.makedirs(dst_dir, exist_ok=True)

for name in ["custom_model.h5", "custom_model.tflite", "custom_model.tflite.summary.txt"]:
    src = os.path.join(src_dir, name)
    if os.path.exists(src):
        print("Copying:", src, "->", os.path.join(dst_dir, name))
        shutil.copy2(src, os.path.join(dst_dir, name))
    else:
        print("[WARN] Missing:", src)